In [20]:
import os
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
import numpy as np
import editdistance

In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [22]:
base_dir = os.getcwd()

ground_truth_path = os.path.join(base_dir, '/kaggle/input/balinese/data/balinese_transliteration_train.txt') 
images_dir = os.path.join(base_dir, '/kaggle/input/balinese/data/balinese_word_train')

In [23]:
filenames = []
labels = []

# Read the ground truth text file
with open(ground_truth_path, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()
        if line:  # Ensure the line is not empty
            parts = line.split(';')
            if len(parts) == 2:
                filename, label = parts
                filenames.append(filename)
                labels.append(label)
            else:
                print(f"Skipping malformed line: {line}")

# Create a DataFrame from the data
data = pd.DataFrame({
    'filename': filenames,
    'label': labels
})

# Display the DataFrame
data.head()


,filename,label
0,train1.png,kagastu
1,train2.png,","
2,train3.png,gelak
3,train4.png,ancak
4,train5.png,","


In [24]:
# Build a character vocabulary from the labels
all_text = ''.join(data['label'])  # Concatenate all labels into one string
chars = sorted(list(set(all_text)))  # Extract unique characters and sort them

# Create character to index mapping, starting from index 1
char_to_idx = {char: idx + 1 for idx, char in enumerate(chars)}  # Start indexing from 1

# Add special tokens
char_to_idx['<PAD>'] = 0  # Add padding token at index 0
char_to_idx['<UNK>'] = len(char_to_idx)  # Add unknown token
char_to_idx['<SOS>'] = len(char_to_idx)  # Add start-of-sequence token
char_to_idx['<EOS>'] = len(char_to_idx)  # Add end-of-sequence token

# Create index to character mapping
idx_to_char = {idx: char for char, idx in char_to_idx.items()}

# Update vocabulary size
vocab_size = len(char_to_idx)

print(f"Vocabulary size: {vocab_size}")


Vocabulary size: 58


In [25]:
# print("Character to Index Mapping:")
# for char, idx in char_to_idx.items():
#     print(f"'{char}': {idx}")


In [26]:
empty_labels = data[data['label'].str.strip() == '']
print(f"Number of empty labels in training data: {len(empty_labels)}")

Number of empty labels in training data: 0


In [27]:
# def encode_label(label, char_to_idx, max_length):
#     encoded = [char_to_idx[char] for char in label]
#     # Pad the encoded label
#     padded = encoded + [char_to_idx['<PAD>']] * (max_length - len(encoded))
#     return padded

def encode_label(label, char_to_idx, max_length):
    encoded = (
        [char_to_idx['<SOS>']] +
        [char_to_idx.get(char, char_to_idx['<UNK>']) for char in label] +
        [char_to_idx['<EOS>']]
    )
    padded = encoded + [char_to_idx['<PAD>']] * (max_length - len(encoded))
    return padded

# add padding to all labels to ensure all labels have same length, use index of character to represent the label
# Encode labels as sequences of character indices
max_label_length = max(len(label) for label in data['label']) + 2  # +2 for <SOS> and <EOS>

data['encoded_label'] = data['label'].apply(lambda x: encode_label(x, char_to_idx, max_label_length))
data['label_length'] = data['label'].apply(len)



In [28]:
# Split the dataset into training and validation sets
train_data, val_data = train_test_split(data, test_size=0.1, random_state=42)

In [29]:
# Define a custom dataset class
# Image data
class BalineseDataset(Dataset):
    def __init__(self, data, images_dir, transform=None):
        self.data = data.reset_index(drop=True)
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """given index, return corresponding image and label"""
        img_name = self.data.loc[idx, 'filename']
        label = self.data.loc[idx, 'encoded_label']
        label_length = self.data.loc[idx, 'label_length']  
        
        img_path = os.path.join(self.images_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(label, dtype=torch.long)
        return image, label, torch.tensor(label_length, dtype=torch.long)


In [30]:
# Define image transformations
transform = transforms.Compose([
    transforms.Resize((64, 256)),  # Resize images to a fixed size
    transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5),  # Mean for R, G, B channels
                            (0.5, 0.5, 0.5))  # Std for R, G, B channels  # Normalize images
])

# Create dataset instances
train_dataset = BalineseDataset(train_data, images_dir, transform=transform)
val_dataset = BalineseDataset(val_data, images_dir, transform=transform)


In [31]:
# Create data loaders
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)


In [32]:
# Define the Encoder
class EncoderCNN(nn.Module):
    def __init__(self, encoded_image_size=14):
        super(EncoderCNN, self).__init__()
        self.enc_image_size = encoded_image_size

        # Pretrained ResNet model
        resnet = models.resnet18(pretrained=True)
        modules = list(resnet.children())[:-2]  # Remove the last two layers (avgpool and fc)
        self.resnet = nn.Sequential(*modules)

        # Resize the encoded images to a fixed size
        self.adaptive_pool = nn.AdaptiveAvgPool2d((encoded_image_size, encoded_image_size))

    def forward(self, images):
        features = self.resnet(images)  # [batch_size, 512, h/32, w/32]
        features = self.adaptive_pool(features)  # [batch_size, 512, encoded_image_size, encoded_image_size]
        # Flatten the features to (batch_size, num_pixels, encoder_dim)
        features = features.permute(0, 2, 3, 1)  # [batch_size, encoded_image_size, encoded_image_size, 512]
        features = features.view(features.size(0), -1, features.size(-1))  # [batch_size, num_pixels, encoder_dim]
        return features


In [33]:
# Define the Attention Mechanism
class Attention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super(Attention, self).__init__()
        self.encoder_att = nn.Linear(encoder_dim, attention_dim)  # Linear layer to transform encoder output
        self.decoder_att = nn.Linear(decoder_dim, attention_dim)  # Linear layer to transform decoder hidden state
        self.full_att = nn.Linear(attention_dim, 1)  # Linear layer to compute attention scores
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)  # Softmax to compute weights

    def forward(self, encoder_out, decoder_hidden):
        # encoder_out: [batch_size, num_pixels, encoder_dim]
        # decoder_hidden: [batch_size, decoder_dim]
        att1 = self.encoder_att(encoder_out)  # [batch_size, num_pixels, attention_dim]
        att2 = self.decoder_att(decoder_hidden)  # [batch_size, attention_dim]
        att = self.full_att(self.relu(att1 + att2.unsqueeze(1))).squeeze(2)  # [batch_size, num_pixels]
        alpha = self.softmax(att)  # [batch_size, num_pixels]
        # convex vector
        attention_weighted_encoding = (encoder_out * alpha.unsqueeze(2)).sum(dim=1)  # [batch_size, encoder_dim]
        return attention_weighted_encoding, alpha


In [34]:
# Define the Decoder
class DecoderRNN(nn.Module):
    def __init__(self, attention_dim, embed_dim, decoder_dim, vocab_size, encoder_dim=512, teacher_forcing_ratio=0.5):
        super(DecoderRNN, self).__init__()
        self.attention = Attention(encoder_dim, decoder_dim, attention_dim) 
        self.embedding = nn.Embedding(vocab_size, embed_dim)  
        self.dropout = nn.Dropout(p=0.5)
        self.decode_step = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim, bias=True)  # LSTMCell
        self.init_h = nn.Linear(encoder_dim, decoder_dim)  # Linear layer to initialize hidden state
        self.init_c = nn.Linear(encoder_dim, decoder_dim)  # Linear layer to initialize cell state
        self.f_beta = nn.Linear(decoder_dim, encoder_dim)  # Linear layer to create a gating scalar
        self.sigmoid = nn.Sigmoid()
        self.fc = nn.Linear(decoder_dim, vocab_size)  # Linear layer to find scores over vocabulary
        self.teacher_forcing_ratio = teacher_forcing_ratio 
        self.init_weights()  

    def init_weights(self):
        """Initialize weights."""
        self.embedding.weight.data.uniform_(-0.1, 0.1)
        self.fc.bias.data.fill_(0)
        self.fc.weight.data.uniform_(-0.1, 0.1)

    def init_hidden_state(self, encoder_out):
        """Create the initial hidden and cell states for the decoder's LSTM based on the encoded images."""
        mean_encoder_out = encoder_out.mean(dim=1)
        h = self.init_h(mean_encoder_out)  # [batch_size, decoder_dim]
        c = self.init_c(mean_encoder_out)
        return h, c

    def forward(self, encoder_out, encoded_captions, caption_lengths):
        """
        Forward propagation.
        :param encoder_out:      [batch_size, num_pixels, encoder_dim]
        :param encoded_captions: [batch_size, max_caption_length]
        :param caption_lengths:  [batch_size, 1]
        """
        caption_lengths, sort_ind = caption_lengths.squeeze(1).sort(dim=0, descending=True)
        encoder_out = encoder_out[sort_ind]
        encoded_captions = encoded_captions[sort_ind]

        embeddings = self.embedding(encoded_captions)
        h, c = self.init_hidden_state(encoder_out)

        decode_lengths = (caption_lengths - 1).tolist()
        max_decode_length = max(decode_lengths)

        batch_size = encoder_out.size(0)
        vocab_size = self.fc.out_features
        predictions = torch.zeros(batch_size, max_decode_length, vocab_size, device=encoder_out.device)
        alphas = torch.zeros(batch_size, max_decode_length, encoder_out.size(1), device=encoder_out.device)

        prev_tokens = encoded_captions[:, 0].detach().clone()

        for t in range(max_decode_length):
            batch_size_t = sum([l > t for l in decode_lengths])

            # Attention
            attention_weighted_encoding, alpha = self.attention(
                encoder_out[:batch_size_t],
                h[:batch_size_t]
            )
            gate = self.sigmoid(self.f_beta(h[:batch_size_t]))
            attention_weighted_encoding = gate * attention_weighted_encoding

            # teacher forcing 
            use_teacher_forcing = (torch.rand(1).item() < self.teacher_forcing_ratio)
            if use_teacher_forcing:
                current_input = embeddings[:batch_size_t, t, :]
            else:
                # Also detach any predicted indices so they are not part of gradient
                current_input = self.embedding(prev_tokens[:batch_size_t].detach())

            # LSTM 
            h_next, c_next = self.decode_step(
                torch.cat([current_input, attention_weighted_encoding], dim=1),
                (h[:batch_size_t], c[:batch_size_t])
            )

            # Predict the next token
            preds = self.fc(self.dropout(h_next))
            predictions[:batch_size_t, t, :] = preds
            alphas[:batch_size_t, t, :] = alpha

            # Update prev_tokens
            _, next_tokens = preds.max(dim=1)
            prev_tokens_ = prev_tokens.clone()
            prev_tokens_[:batch_size_t] = next_tokens.detach()
            prev_tokens = prev_tokens_

            # update h, c for next step
            # h[:batch_size_t] = h_next
            # c[:batch_size_t] = c_next
            h_new = torch.zeros_like(h)
            c_new = torch.zeros_like(c)
            h_new[:batch_size_t] = h_next
            c_new[:batch_size_t] = c_next
            h, c = h_new, c_new

        return predictions, encoded_captions, decode_lengths, alphas, sort_ind



In [35]:
# Instantiate the encoder and decoder
encoder = EncoderCNN()
decoder = DecoderRNN(
    attention_dim=256,
    embed_dim=256,
    decoder_dim=512,
    vocab_size=vocab_size,
    encoder_dim=512,
    teacher_forcing_ratio=0.5 
)

# Move models to device
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = encoder.to(device)
decoder = decoder.to(device)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [36]:
# Define loss function and optimizers
criterion = nn.CrossEntropyLoss(ignore_index=char_to_idx['<PAD>'])
encoder_optimizer = optim.Adam(encoder.parameters(), lr=1e-4)
decoder_optimizer = optim.Adam(decoder.parameters(), lr=4e-4)

In [37]:
class ImageCaptioningTrainer:
    def __init__(self, encoder, decoder, 
                 criterion, encoder_optimizer, decoder_optimizer, 
                 train_loader, val_loader, device, 
                 char_to_idx, idx_to_char, max_label_length):
        self.encoder = encoder.to(device)
        self.decoder = decoder.to(device)
        self.criterion = criterion
        self.encoder_optimizer = encoder_optimizer
        self.decoder_optimizer = decoder_optimizer
        self.train_loader = train_loader
        self.device = device
        self.val_loader = val_loader
        self.char_to_idx = char_to_idx
        self.idx_to_char = idx_to_char
        self.max_label_length = max_label_length

    def fit(self, num_epochs):
        for epoch in range(num_epochs):
            print(f'Epoch {epoch + 1}/{num_epochs}')
            train_loss, train_cer = self.train_one_epoch()
            val_loss, val_cer = self.validate_one_epoch()
            print(f'Epoch [{epoch +1}/{num_epochs}] - '
                f'Train Loss: {train_loss:.4f}, Train CER: {train_cer:.4f} - '
                f'Val Loss: {val_loss:.4f}, Val CER: {val_cer:.4f}\n')

    def train_one_epoch(self):
        self.encoder.train()
        self.decoder.train()
        running_loss = 0.0
        total_edit_distance = 0
        total_ref_length = 0
        total_samples = 0

        for batch_idx, (images, labels, label_lengths) in enumerate(self.train_loader):
            images = images.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)
            label_lengths = label_lengths.to(self.device, non_blocking=True)

            # Zero the gradients
            self.encoder_optimizer.zero_grad()
            self.decoder_optimizer.zero_grad()

            # Forward pass
            encoder_out = self.encoder(images)
            caption_lengths = torch.tensor(
                [self.max_label_length] * labels.size(0)
                ).unsqueeze(1).to(self.device)
            outputs, encoded_captions, decode_lengths, alphas, sort_ind = self.decoder(
                encoder_out, labels, caption_lengths
            )

            # Reshape targets
            targets = encoded_captions[:, 1:] # remove <SOS>

            # Flatten outputs and targets for loss computation
            outputs_flat = outputs.view(-1, self.decoder.fc.out_features)
            targets_flat = targets.contiguous().view(-1)

            # Compute loss
            loss = self.criterion(outputs_flat, targets_flat)
            loss.backward()

            # Update weights
            self.decoder_optimizer.step()
            self.encoder_optimizer.step()

            running_loss += loss.item()

            # Compute CER for the batch
            batch_size = labels.size(0)
            _, preds_flat = torch.max(outputs_flat, dim=1)
            preds_seq = preds_flat.view(batch_size, -1)

            for i in range(batch_size):
                pred_indices = preds_seq[i].detach().cpu().numpy()
                target_indices = targets[i].detach().cpu().numpy()

                # Remove padding tokens
                mask = target_indices != self.char_to_idx['<PAD>']
                pred_indices = pred_indices[mask]
                target_indices = target_indices[mask]

                # Decode indices to characters
                pred_chars = [self.idx_to_char.get(idx, '') for idx in pred_indices]
                target_chars = [self.idx_to_char.get(idx, '') for idx in target_indices]

                # Join characters to form strings
                pred_str = ''.join(pred_chars)
                target_str = ''.join(target_chars)

                # Compute edit distance
                edit_dist = editdistance.eval(pred_str, target_str)
                total_edit_distance += edit_dist
                total_ref_length += len(target_str)
    
            total_samples += batch_size

            if (batch_idx + 1) % 50 == 0 or (batch_idx + 1) == len(self.train_loader):
                print(f'Batch {batch_idx + 1}/{len(self.train_loader)} - Loss: {loss.item():.4f}')

        avg_loss = running_loss / len(self.train_loader)
        # compute the "global" CER
        if total_ref_length > 0:
            avg_cer = total_edit_distance / total_ref_length
        else:
            avg_cer = 0.0  # avoid division by zero if something is off
        
        return avg_loss, avg_cer

    def validate_one_epoch(self):
        self.encoder.eval()
        self.decoder.eval()
        running_loss = 0.0
        total_edit_distance = 0
        total_ref_length = 0
        total_samples = 0 

        with torch.no_grad():
            for batch_idx, (images, labels, label_lengths) in enumerate(self.val_loader):
                images = images.to(self.device, non_blocking=True)
                labels = labels.to(self.device, non_blocking=True)
                label_lengths = label_lengths.to(self.device, non_blocking=True)

                # Forward pass
                encoder_out = self.encoder(images)
                caption_lengths = torch.tensor(
                    [self.max_label_length] * labels.size(0)
                ).unsqueeze(1).to(self.device)
    
                outputs, encoded_captions, decode_lengths, alphas, sort_ind = self.decoder(
                    encoder_out, labels, caption_lengths
                )

                # Reshape targets
                targets = encoded_captions[:, 1:]

                # Flatten outputs and targets for loss computation
                outputs_flat = outputs.view(-1, self.decoder.fc.out_features)
                targets_flat = targets.contiguous().view(-1)

                # Compute loss
                loss = self.criterion(outputs_flat, targets_flat)
                running_loss += loss.item()

                # Compute *global* style edit distance
                batch_size = labels.size(0)
                _, preds_flat = torch.max(outputs_flat, dim=1)
                preds_seq = preds_flat.view(batch_size, -1)

                for i in range(batch_size):
                    pred_indices = preds_seq[i].detach().cpu().numpy()
                    target_indices = targets[i].detach().cpu().numpy()

                    # Remove padding tokens
                    mask = target_indices != self.char_to_idx['<PAD>']
                    pred_indices = pred_indices[mask]
                    target_indices = target_indices[mask]

                    # Decode indices to characters
                    pred_chars = [self.idx_to_char.get(idx, '') for idx in pred_indices]
                    target_chars = [self.idx_to_char.get(idx, '') for idx in target_indices]

                    # Join characters to form strings
                    pred_str = ''.join(pred_chars)
                    target_str = ''.join(target_chars)

                    # Compute edit distance
                    edit_dist = editdistance.eval(pred_str, target_str)
                    total_edit_distance += edit_dist
                    total_ref_length += len(target_str)

                    # Print a few samples
                    if batch_idx == 0 and i < 3: 
                        print(f"Sample {i + 1}:")
                        print(f"Predicted: {pred_str}")
                        print(f"Target   : {target_str}\n")

                total_samples += batch_size

        avg_loss = running_loss / len(self.val_loader)
        
        if total_ref_length > 0:
            avg_cer = total_edit_distance / total_ref_length
        else:
            avg_cer = 0.0  # avoid division by zero if something is off
        
        return avg_loss, avg_cer

In [38]:
trainer = ImageCaptioningTrainer(
    encoder=encoder,
    decoder=decoder,
    criterion=criterion,
    encoder_optimizer=encoder_optimizer,
    decoder_optimizer=decoder_optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    char_to_idx=char_to_idx,
    idx_to_char=idx_to_char,
    max_label_length=max_label_length
)

# trainer = trainer.to(device)
num_epochs = 30
trainer.fit(num_epochs)

Epoch 1/30
Batch 50/423 - Loss: 2.1380
Batch 250/423 - Loss: 1.9519
Batch 300/423 - Loss: 1.6510
Batch 350/423 - Loss: 1.6808
Batch 400/423 - Loss: 1.6994
Batch 423/423 - Loss: 1.8474
Sample 1:
Predicted: angenn<EOS>
Target   : anggon<EOS>

Sample 2:
Predicted: tuuununun<EOS><EOS><EOS><EOS>
Target   : kaputusaning<EOS>

Sample 3:
Predicted: ,<EOS>
Target   : ,<EOS>

Epoch [1/30] - Train Loss: 1.9775, Train CER: 0.6425 - Val Loss: 1.4433, Val CER: 0.4431

Epoch 2/30
Batch 50/423 - Loss: 1.3944
Batch 100/423 - Loss: 1.3820
Batch 150/423 - Loss: 1.4286
Batch 200/423 - Loss: 1.4226
Batch 250/423 - Loss: 1.1438
Batch 300/423 - Loss: 1.3187
Batch 350/423 - Loss: 1.0488
Batch 400/423 - Loss: 1.2261
Batch 423/423 - Loss: 1.1200
Sample 1:
Predicted: anggang
Target   : anggon<EOS>

Sample 2:
Predicted: pupusuning<EOS>g<EOS>
Target   : kaputusaning<EOS>

Sample 3:
Predicted: ,<EOS>
Target   : ,<EOS>

Epoch [2/30] - Train Loss: 1.2190, Train CER: 0.3559 - Val Loss: 0.9689, Val CER: 0.2894

Epoch 3

In [39]:
# Paths to test data
test_ground_truth_path = os.path.join(base_dir, '/kaggle/input/balinese/data/balinese_transliteration_test.txt')
test_images_dir = os.path.join(base_dir, '/kaggle/input/balinese/data/balinese_word_test')

test_filenames = []
test_labels = []

# Read the ground truth text file for the test set
with open(test_ground_truth_path, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()
        if line:
            parts = line.split(';')
            if len(parts) == 2:
                filename, label = parts
                test_filenames.append(filename)
                test_labels.append(label)
            else:
                print(f"Skipping malformed line: {line}")

# Create a DataFrame from the test data
test_data = pd.DataFrame({
    'filename': test_filenames,
    'label': test_labels
})


In [40]:
test_chars = set(''.join(test_data['label']))
unknown_chars = test_chars - set(char_to_idx.keys())
print(f"Unknown characters in test labels: {unknown_chars}")

Unknown characters in test labels: {'H'}


In [41]:
empty_test_labels = test_data[test_data['label'].str.strip() == '']
print(f"Number of empty labels in test data: {len(empty_test_labels)}")

Number of empty labels in test data: 0


In [42]:
max_label_length_test = max(len(lbl) for lbl in test_data['label']) + 2

def encode_label_test(label, char_to_idx, max_length):
    encoded = (
        [char_to_idx['<SOS>']] +
        [char_to_idx.get(ch, char_to_idx['<UNK>']) for ch in label] +
        [char_to_idx['<EOS>']]
    )
    # Truncate or pad to max_length
    if len(encoded) > max_length:
        encoded = encoded[:max_length]
    else:
        encoded += [char_to_idx['<PAD>']] * (max_length - len(encoded))
    return encoded

# Create encoded_label and label_length columns in test_data
test_data['encoded_label'] = test_data['label'].apply(lambda x: encode_label_test(x, char_to_idx, max_label_length_test))
test_data['label_length'] = test_data['label'].apply(len)

In [43]:
test_transform = transforms.Compose([
    transforms.Resize((64, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

test_dataset = BalineseDataset(test_data, test_images_dir, transform=test_transform)

batch_size = 32

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

In [44]:
import numpy as np

def inference(encoder, decoder, test_loader, device, char_to_idx, idx_to_char, max_seq_length, test_data):
    """
    Greedy decoding inference on test data that has ground-truth labels.
    
    Args:
        encoder (nn.Module): Encoder CNN.
        decoder (nn.Module): Decoder RNN with attention.
        test_loader (DataLoader): yields (images, labels, label_lengths).
        device (torch.device): 'cuda' or 'cpu'.
        char_to_idx (dict): Mapping from character to index.
        idx_to_char (dict): Mapping from index to character.
        max_seq_length (int): Maximum decoding steps.
        test_data (DataFrame): Reference to get filenames or additional info.
    
    Returns:
        results (list of dict):
            [
                {
                    "image_filename": str,
                    "predicted_caption": str,
                    "ground_truth_caption": str
                },
                ...
            ]
    """
    encoder.eval()
    decoder.eval()
    results = []

    # We'll need the <EOS> token index, in case we find it in predictions
    eos_idx = char_to_idx['<EOS>']

    with torch.no_grad():
        for batch_idx, (images, labels, label_lengths) in enumerate(test_loader):
            # Move data to GPU if available
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            # label_lengths = label_lengths.to(device, non_blocking=True)  # Not strictly needed for greedy decode

            batch_size = images.size(0)

            # Encode images
            encoder_out = encoder(images)  # [batch_size, num_pixels, encoder_dim]

            # Initialize LSTM hidden/cell
            h, c = decoder.init_hidden_state(encoder_out)

            # Current input token: <SOS> for all in the batch
            inputs = torch.full(
                (batch_size,),
                fill_value=char_to_idx['<SOS>'],
                dtype=torch.long,
                device=device
            )

            # We'll store predicted tokens at each step
            all_preds = []

            for _ in range(max_seq_length):
                # 1) Embed the current token
                embeddings = decoder.embedding(inputs)  # [batch_size, embed_dim]

                # 2) Attention over encoder outputs
                attention_weighted_encoding, alpha = decoder.attention(encoder_out, h)

                # 3) Gating
                gate = decoder.sigmoid(decoder.f_beta(h))
                attention_weighted_encoding = gate * attention_weighted_encoding

                # 4) LSTM step
                h, c = decoder.decode_step(
                    torch.cat([embeddings, attention_weighted_encoding], dim=1),
                    (h, c)
                )

                # 5) Predict next token
                preds = decoder.fc(decoder.dropout(h))  # [batch_size, vocab_size]
                _, preds_idx = preds.max(dim=1)

                # 6) Append predictions for each sample
                all_preds.append(preds_idx.cpu().numpy())

                # 7) Next input is the predicted token
                inputs = preds_idx

            # Convert from shape [max_seq_length, batch_size] to [batch_size, max_seq_length]
            all_preds = np.array(all_preds).T

            # For each sample in the batch
            for i in range(batch_size):
                pred_indices = all_preds[i]

                # Stop at <EOS> if present
                if eos_idx in pred_indices:
                    first_eos = np.where(pred_indices == eos_idx)[0][0]
                    pred_indices = pred_indices[:first_eos]

                pred_chars = [idx_to_char.get(idx, '') for idx in pred_indices]
                pred_str = ''.join(pred_chars)

                # Reconstruct ground truth label (removing <SOS> and <EOS>)
                label_indices = labels[i].detach().cpu().numpy()
                label_indices = label_indices[1:]  # drop <SOS>
                if eos_idx in label_indices:
                    first_eos_gt = np.where(label_indices == eos_idx)[0][0]
                    label_indices = label_indices[:first_eos_gt]
                else:
                    # Also remove <PAD> if any
                    label_indices = label_indices[label_indices != char_to_idx['<PAD>']]

                label_chars = [idx_to_char.get(idx, '') for idx in label_indices]
                label_str = ''.join(label_chars)

                # Grab the filename from test_data
                global_idx = batch_idx * batch_size + i
                image_filename = test_data.iloc[global_idx]['filename']

                results.append({
                    'image_filename': image_filename,
                    'predicted_caption': pred_str,
                    'ground_truth_caption': label_str
                })

    return results

In [45]:
# Run inference on the test set
test_results = inference(
    encoder=encoder,
    decoder=decoder,
    test_loader=test_loader,
    device=device,
    char_to_idx=char_to_idx,
    idx_to_char=idx_to_char,
    max_seq_length=max_label_length_test,  # or some other limit
    test_data=test_data
)

# Inspect the first few results
for r in test_results[:5]:
    print("Image:", r['image_filename'])
    print("Predicted:", r['predicted_caption'])
    print("Ground Truth:", r['ground_truth_caption'])
    print()


Image: test1.png
Predicted: ,
Ground Truth: ,

Image: test2.png
Predicted: biak
Ground Truth: biakta

Image: test3.png
Predicted: eattah
Ground Truth: ngantah

Image: test4.png
Predicted: arica
Ground Truth: sarira

Image: test5.png
Predicted: yu
Ground Truth: yu



In [46]:
import editdistance

def calculate_global_cer(test_results):
    """
    Computes the global CER = (sum of all edit distances) / (sum of all reference lengths).
    test_results should be a list of dicts with:
      {
        'ground_truth_caption': str,
        'predicted_caption': str,
        ...
      }
    Returns:
      A float representing the global CER.
    """
    total_edit_distance = 0
    total_reference_length = 0

    for result in test_results:
        reference  = result['ground_truth_caption']
        hypothesis = result['predicted_caption']
        
        dist = editdistance.eval(reference, hypothesis)
        total_edit_distance     += dist
        total_reference_length  += len(reference)

    if total_reference_length > 0:
        cer = total_edit_distance / total_reference_length
    else:
        cer = 0.0

    return cer

In [47]:
# Compute global CER
global_cer = calculate_global_cer(test_results)
print(f"Global CER on Test Set: {global_cer:.4f}")

# Add per-sample CER to each entry in test_results so we can sort by it
for i, result in enumerate(test_results):
    ref = result['ground_truth_caption']
    hyp = result['predicted_caption']
    # avoid zero-division if ref is empty
    if len(ref) == 0:
        sample_cer = 0.0 if len(hyp) == 0 else float('inf')
    else:
        sample_cer = editdistance.eval(ref, hyp) / len(ref)
    result['cer'] = sample_cer


Global CER on Test Set: 0.1998


In [48]:
sorted_results = sorted(test_results, key=lambda x: x['cer'], reverse=True)

N = 10
print(f"\nTop {N} samples with highest CER:")
for i in range(min(N, len(sorted_results))):
    r = sorted_results[i]
    print(f"Sample {i+1}:")
    print(f"Image Filename: {r['image_filename']}")
    print(f"CER: {r['cer']:.4f}")
    print(f"Ground Truth : {r['ground_truth_caption']}")
    print(f"Predicted    : {r['predicted_caption']}\n")



Top 10 samples with highest CER:
Sample 1:
Image Filename: test3029.png
CER: 11.0000
Ground Truth : .
Predicted    : carik-agung

Sample 2:
Image Filename: test4934.png
CER: 11.0000
Ground Truth : .
Predicted    : carik-agung

Sample 3:
Image Filename: test10070.png
CER: 10.0000
Ground Truth : .
Predicted    : iaannksang

Sample 4:
Image Filename: test1330.png
CER: 8.0000
Ground Truth : .
Predicted    : angsanya

Sample 5:
Image Filename: test2690.png
CER: 5.0000
Ground Truth : B
Predicted    : jakik

Sample 6:
Image Filename: test3142.png
CER: 5.0000
Ground Truth : i
Predicted    : shang

Sample 7:
Image Filename: test8305.png
CER: 5.0000
Ground Truth : 7
Predicted    : liang

Sample 8:
Image Filename: test585.png
CER: 4.0000
Ground Truth : 9
Predicted    : yang

Sample 9:
Image Filename: test726.png
CER: 4.0000
Ground Truth : 6
Predicted    : cing

Sample 10:
Image Filename: test1731.png
CER: 4.0000
Ground Truth : .
Predicted    : pama



In [49]:
# Additionally, print some random samples to get a general idea of model performance
import random

M = 5  # Number of random samples to display
random_samples = random.sample(test_results, M)
print(f"\nRandom {M} samples from Test Set:")
for i, result in enumerate(random_samples):
    print(f"Sample {i + 1}:")
    print(f"Image Filename: {result['image_filename']}")
    print(f"CER: {result['cer']:.4f}")
    print(f"Ground Truth Caption: {result['ground_truth_caption']}")
    print(f"Predicted Caption  : {result['predicted_caption']}\n")


Random 5 samples from Test Set:
Sample 1:
Image Filename: test7359.png
CER: 0.3333
Ground Truth Caption: mretaning
Predicted Caption  : kratening

Sample 2:
Image Filename: test9798.png
CER: 0.0000
Ground Truth Caption: ,
Predicted Caption  : ,

Sample 3:
Image Filename: test2277.png
CER: 0.2500
Ground Truth Caption: manuknia
Predicted Caption  : manukniang

Sample 4:
Image Filename: test7302.png
CER: 0.1250
Ground Truth Caption: patraksi
Predicted Caption  : patraksia

Sample 5:
Image Filename: test6356.png
CER: 0.2500
Ground Truth Caption: hana
Predicted Caption  : han

